In [23]:
import numpy as np
from sympy import Matrix

# ---------------------------------------------------------
# 1. LLL reduction algorithm (integer-preserving, column-based)
# ---------------------------------------------------------

def lll_reduce(B, delta=0.75):
    B = Matrix(B)
    n = B.cols

    def gs(B):
        m = B.rows
        U = [B[:, i] for i in range(n)]
        U_star = [U[0]]
        mu = [[0]*n for _ in range(n)]
        mu[0][0] = 1

        for i in range(1, n):
            proj = Matrix.zeros(m, 1)
            for j in range(i):
                proj += (U[i].dot(U_star[j]) / U_star[j].dot(U_star[j])) * U_star[j]
            U_star.append(U[i] - proj)

            for j in range(i):
                mu[i][j] = U[i].dot(U_star[j]) / U_star[j].dot(U_star[j])
            mu[i][i] = 1

        return U_star, mu

    U_star, mu = gs(B)
    k = 1

    while k < n:
        # Size reduction
        for j in range(k-1, -1, -1):
            q = round(mu[k][j])
            if q != 0:
                B[:, k] -= q * B[:, j]

        U_star, mu = gs(B)

        # Lovász condition
        if U_star[k].dot(U_star[k]) >= (delta - mu[k][k-1]**2) * U_star[k-1].dot(U_star[k-1]):
            k += 1
        else:
            B.col_swap(k, k-1)
            U_star, mu = gs(B)
            k = max(k-1, 1)

    return B

# ---------------------------------------------------------
# 2. Shortest vector finder (Due to SymPy syntax, scans columns but that is mathematically correct)
# ---------------------------------------------------------

def find_shortest_vector(B):
    m, n = B.shape
    shortest = None
    shortest_norm = float('inf')

    for i in range(n):  # columns, not rows
        v = B[:, i]
        norm = float(v.norm())
        if norm < shortest_norm:
            shortest_norm = norm
            shortest = v

    return shortest

# ---------------------------------------------------------
# 3. Dual lattice basis for LCG (Knuth/Tezuka)
# ---------------------------------------------------------

def build_lcg_dual_basis(a, m, s):
    """
    Dual lattice basis L_s^* for LCG X_{n+1} = a X_n mod m
    in s dimensions (Knuth/Tezuka).
    """
    B = []
    for i in range(s):
        row = [0] * s
        if i == 0:
            row[0] = m
        else:
            row[0] = pow(int(a), i, int(m))
        row[i] = 1
        B.append(row)
    return B

# ---------------------------------------------------------
# 4. Hermite constants γ_s for Tezuka normalization
# ---------------------------------------------------------

HERMITE_CONSTANTS = {
    2: 2 / np.sqrt(3),
    3: (2/np.sqrt(3))**(3/2),
    4: 2.0,
    5: 2.0**(5/4),
    6: 2.0**(3/2),
    7: 2.0**(7/4)
}

# ---------------------------------------------------------
# 5. Tezuka-normalized spectral score (Real Metric)
# ---------------------------------------------------------

def tezuka_spectral_score(shortest_vector, m, s):
    v = np.array(shortest_vector, dtype=float).flatten()
    d_s = np.linalg.norm(v)

    gamma_s = HERMITE_CONSTANTS[s]
    best_possible = np.sqrt(gamma_s) * (m ** (1.0 / s))

    return (1.0 / d_s) / best_possible

# ---------------------------------------------------------
# 5? Placeholder score (kept for documentation)
# ---------------------------------------------------------

# def spectral_score_from_vector(shortest_vector):
#     v = np.array(shortest_vector, dtype=float).flatten()
#     return np.linalg.norm(v)
# NOTE: This was the placeholder score used during early development.
#       Uncomment this and comment out the Tezuka score in the main loop
#       if you want to run the placeholder version.

# ---------------------------------------------------------
# 6. MAIN EXECUTION BLOCK
# ---------------------------------------------------------

def run_spectral_test_lcg(a, m):
    for s in range(2, 8):  # dimensions 2..7
        print(f"\n=== Dimension {s} ===")

        # 1. Build dual basis
        B_list = build_lcg_dual_basis(a, m, s)
        B = Matrix(B_list)

        # 2. LLL reduction
        B_reduced = lll_reduce(B)

        # 3. Shortest vector
        shortest = find_shortest_vector(B_reduced)
        
        # 4. Tezuka spectral score (REAL METRIC UNLESS CHANGED)
        score = tezuka_spectral_score(shortest, m, s)

        # 5. Output
        print("Shortest vector:", shortest)
        print("Tezuka spectral score:", score)

# ---------------------------------------------------------
# Run the test with your LCG parameters
# ---------------------------------------------------------

a = 16807   # example multiplier
m = 2**31 - 1  # example modulus

run_spectral_test_lcg(a, m)



=== Dimension 2 ===
Shortest vector: Matrix([[0], [1]])
Tezuka spectral score: 2.008169575895561e-05

=== Dimension 3 ===
Shortest vector: Matrix([[0], [1], [0]])
Tezuka spectral score: 0.0006958324615292189

=== Dimension 4 ===
Shortest vector: Matrix([[0], [1], [0], [0]])
Tezuka spectral score: 0.0032847516224672173

=== Dimension 5 ===
Shortest vector: Matrix([[0], [1], [0], [0], [0]])
Tezuka spectral score: 0.008820034413369328

=== Dimension 6 ===
Shortest vector: Matrix([[0], [1], [0], [0], [0], [0]])
Tezuka spectral score: 0.016554110850648757

=== Dimension 7 ===
Shortest vector: Matrix([[0], [1], [0], [0], [0], [0], [0]])
Tezuka spectral score: 0.02532012911106336
